In [ ]:
import os, glob, json
import networkx as nx

In [ ]:
# Get the JSON files
json_paths = glob.glob("../bddl/generated_data/transition_map/tm_jsons/*.json")
data = []
for jp in json_paths:
    with open(jp) as f:
        data.append(json.load(f))
transitions = [rule for rules in data for rule in rules]

In [ ]:
transitions

In [ ]:
import collections

c = collections.Counter(x["rule_name"] for x in transitions)
[t for t in transitions if c[t["rule_name"]] > 1]

In [ ]:
# Build the transition network
G = nx.DiGraph()
for transition in transitions:
    rule_name = transition["rule_name"]
    G.add_node(rule_name, type="rule")
    for input_obj in transition["input_objects"].keys():
        G.add_node(input_obj, type="obj")
        G.add_edge(input_obj, rule_name)
    for output_obj in transition["output_objects"].keys():
        G.add_node(output_obj, type="obj")
        G.add_edge(rule_name, output_obj)

In [ ]:
def synset_is_reachable(s, available_objs):
    # If it's already available, then we're good.
    if s in available_objs:
        print(s, "is already available in available objects set")
        return True

    # Otherwise, are there any recipes that I can use to obtain it?
    for recipe, _ in G.in_edges(s):
        print("Considering recipe", recipe, "to obtain", s)
        if all(
            synset_is_reachable(ingredient, available_objs)
            for ingredient, _ in G.in_edges(recipe)
        ):
            print("Recipe", recipe, "was usable to obtain", s)
            return True

    return False


def is_reachable(initial_objs, goal_objs):
    return all(synset_is_reachable(s, initial_objs) for s in goal_objs)

In [ ]:
for f, t in G.subgraph(
    nx.dfs_tree(G.reverse(), "cooked__diced__meat_loaf.n.01").nodes
).edges:
    f_name = f if G.nodes[f]["type"] == "obj" else "%s(%s)" % (f, f)
    t_name = t if G.nodes[t]["type"] == "obj" else "%s(recipe: %s)" % (t, t)
    print(f"{f_name} --> {t_name}")

In [ ]:
synset_is_reachable(
    "cooked__diced__meat_loaf.n.01",
    [
        "diced__vidalia_onion.n.01",
        "brown_sugar.n.01",
        "whole_milk.n.01",
        "breadcrumb.n.01",
        "ground_beef.n.01",
    ],
)